In [1]:
import hashlib
import logging
from cryptography.hazmat.primitives.serialization import Encoding, PublicFormat
from cryptography.x509 import load_der_x509_certificate
from androguard.misc import AnalyzeAPK

logger = logging.getLogger(__name__)

def generate_apk_hash(apk_path: str) -> str:
    sha256 = hashlib.sha256()
    try:
        with open(apk_path, 'rb') as f:
            while chunk := f.read(8192):
                sha256.update(chunk)
        return sha256.hexdigest()
    except Exception as e:
        logger.error(f"Hash generation failed: {e}")
        raise

def extract_metadata(apk_path: str) -> dict:
    try:
        a, _, _ = AnalyzeAPK(apk_path)
        certs = []
        for cert in a.get_certificates():
            try:
                der_data = cert.get_raw() 
                x509_cert = load_der_x509_certificate(der_data)
                public_key = x509_cert.public_key().public_bytes(
                    encoding=Encoding.PEM,
                    format=PublicFormat.SubjectPublicKeyInfo
                ).decode()
                algorithm = x509_cert.signature_algorithm_oid.dotted_string
            except Exception:
                public_key = "Unknown"
                algorithm = "Unknown"
                
            certs.append({
                'public_key': public_key,
                'signature_algorithm': algorithm
            })
        return {
            'package_name': a.get_package(),
            'version': a.get_androidversion_name(),
            'permissions': a.get_permissions(),
            'certificates': certs,
            'activities': a.get_activities()
        }
    except Exception as e:
        logger.error(f"Metadata extraction failed: {e}")
        return {}

In [ ]:
import os

VIRUSTOTAL_API_KEY = "" # deleted for protecting api key
# VIRUSTOTAL_API_KEY = os.getenv("VIRUSTOTAL_API_KEY", "your_default_api_key_here")

log_file = "app_detection.log"
hashes_csv = "hashes.csv"

In [3]:
import requests
import logging
from key import VIRUSTOTAL_API_KEY

logger = logging.getLogger(__name__)

class VirusTotalClient:
    def __init__(self):
        self.base_url = "https://www.virustotal.com/api/v3"
        self.headers = {"x-apikey": VIRUSTOTAL_API_KEY}
        
    def check_hash(self, file_hash: str) -> dict:
        # check file hash against VirusTotal database
        try:
            response = requests.get(
                f"{self.base_url}/files/{file_hash}",
                headers=self.headers
            )
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            logger.error(f"API request failed: {e}")
            return{}

In [5]:
import logging
from key import log_file

def log_setup():
    logging.basicConfig(
        filename=log_file,
        level=logging.INFO,
        format="%(asctime)s - %(levelname)s - %(message)s"
    )
    return logging.getLogger()

In [6]:
import csv
import os
from datetime import datetime

def detection_data(
    scan_name: str,
    apk_path: str,
    apk_hash: str,
    metadata: dict,
    vt_report: dict,
    fake_indicators: list,
    output_csv: str = "csv/appScan.csv"
) -> str:
    file_exists = os.path.isfile(output_csv)

    package_name = metadata.get('package_name', 'N/A')
    version = metadata.get('version', 'N/A')
    permissions_count = len(metadata.get('permissions', []))

    vt_data = vt_report.get('data')
    if vt_data:
        attributes = vt_data.get('attributes', {})
        stats = attributes.get('last_analysis_stats', {})
        vt_clean = stats.get('harmless', 0)
        vt_suspicious = stats.get('suspicious', 0)
        vt_malicious = stats.get('malicious', 0)
    else:
        vt_clean = 0
        vt_suspicious = 0
        vt_malicious = 0

    indicators_str = ", ".join(fake_indicators) if fake_indicators else "None"
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    row = [
        timestamp,
        scan_name,
        apk_path,
        apk_hash,
        package_name,
        version,
        permissions_count,
        vt_clean,
        vt_suspicious,
        vt_malicious,
        indicators_str
    ]

    with open(output_csv, mode='a', newline='', encoding='utf-8') as csv_file:
        writer = csv.writer(csv_file)
        if not file_exists:
            writer.writerow([
                "Timestamp",
                "Scan Name",
                "APK Path",
                "SHA-256 Hash",
                "Package Name",
                "Version",
                "#Permissions",
                "VT Clean",
                "VT Suspicious",
                "VT Malicious",
                "Fake/Clone Indicators"
            ])
        writer.writerow(row)

    return os.path.abspath(output_csv)

In [7]:
from ui import AppDetectorGUI

if __name__ == "__main__":
    app = AppDetectorGUI()
    app.run()